In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install transformers scikit-learn torch lime tqdm seaborn

In [ ]:
import os, time, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

In [ ]:
SEED = 62
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Check use CPU OR GPU

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using: {DEVICE}")

Datasets check From MyDrive

In [ ]:
import os

# Path to the liar folder
liar_folder_path = '/content/drive/MyDrive/datasets/liar'

# Check if the folder exists and list its contents
if os.path.exists(liar_folder_path):
    print(f"Contents of {liar_folder_path}:")
    for item in os.listdir(liar_folder_path):
        print(f"- {item}")
else:
    print(f"The folder {liar_folder_path} does not exist. Please check the path.")

In [ ]:
import os

# Path to the fakenesnet folder
fakenesnet_folder_path = '/content/drive/MyDrive/datasets/fakenewsnet'

# Check if the folder exists and list its contents
if os.path.exists(fakenesnet_folder_path):
    print(f"Contents of {fakenesnet_folder_path}:")
    for item in os.listdir(fakenesnet_folder_path):
        print(f"- {item}")
else:
    print(f"The folder {fakenesnet_folder_path} does not exist. Please check the path.")


Datasets loading

In [ ]:
def load_liar(data_dir):
    # collapse 6 labels to binary, following Alghamdi et al. 2022
    label_map = {
        "false": 0, "barely-true": 0, "pants-fire": 0,
        "true": 1,  "mostly-true": 1, "half-true": 1
    }
    texts, labels = [], []
    for f in ["train.tsv", "valid.tsv", "test.tsv"]:
        path = os.path.join(data_dir, f)
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path, sep="\t", header=None)
        for _, row in df.iterrows():
            lbl = str(row[1]).strip().lower()
            txt = str(row[2]).strip()
            if lbl in label_map:
                texts.append(txt)
                labels.append(label_map[lbl])
    print(f"LIAR: {len(texts)} samples (fake={labels.count(0)}, real={labels.count(1)})")
    return texts, labels

In [ ]:
def load_fakenewsnet(csv_path):
    df = pd.read_csv(csv_path).dropna(subset=["text", "label"])
    texts  = df["text"].tolist()
    labels = df["label"].astype(int).tolist()
    print(f"FakeNewsNet: {len(texts)} samples (fake={labels.count(0)}, real={labels.count(1)})")
    return texts, labels

In [ ]:
def split_data(texts, labels, test_size=0.2):
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=test_size, random_state=SEED, stratify=labels
    )
    print(f"  train={len(X_train)}  test={len(X_test)}")
    return X_train, X_test, y_train, y_test

Evaluate Helper

In [ ]:

def evaluate(y_true, y_pred, name="model"):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n[{name}]")
    print(f"  accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  F1={f1:.4f}")
    print(f"  confusion matrix:\n{confusion_matrix(y_true, y_pred)}")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


def bootstrap_ci(y_true, y_pred, n=1000, alpha=0.05):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    scores = [
        f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0)
        for idx in [np.random.choice(len(y_true), len(y_true), replace=True) for _ in range(n)]
    ]
    lo, hi = np.percentile(scores, [100*alpha/2, 100*(1-alpha/2)])
    print(f"  95% CI: [{lo:.4f}, {hi:.4f}]")
    return lo, hi


def measure_infer_time(model, X_test):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t = time.time()
    model.predict(X_test)
    ms = (time.time() - t) / len(X_test) * 1000
    mem = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    print(f"  inference: {ms:.3f} ms/sample  GPU mem: {mem:.1f} MB")
    return ms

SVM modeling Training and Testing

In [ ]:
class SVMClassifier:

    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            max_features=50000, ngram_range=(1, 2),
            sublinear_tf=True, strip_accents="unicode",
            analyzer="word", stop_words="english"
        )
        self.model  = None
        self.best_C = None

    def _preprocess(self, texts):
        return [t.lower() for t in texts]

    def train(self, X_train, y_train):
        print("\n[SVM] training...")
        t0 = time.time()

        X = normalize(self.vectorizer.fit_transform(self._preprocess(X_train)), norm="l2")
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        best_f1, best_C = -np.inf, None

        for C in [0.01, 0.1, 1.0, 10.0]:
            scores = []
            for tr, val in kf.split(X, y_train):
                clf = LinearSVC(C=C, max_iter=5000, random_state=SEED)
                clf.fit(X[tr], [y_train[i] for i in tr])
                scores.append(f1_score([y_train[i] for i in val], clf.predict(X[val]), average="macro"))
            m = np.mean(scores)
            print(f"  C={C}  F1={m:.4f}")
            if m > best_f1:
                best_f1, best_C = m, C

        self.best_C = best_C
        self.model  = LinearSVC(C=best_C, max_iter=5000, random_state=SEED)
        self.model.fit(X, y_train)
        elapsed = time.time() - t0
        print(f"[SVM] done ({elapsed:.1f}s)  best C={best_C}")
        return elapsed

    def predict(self, X_test):
        X = normalize(self.vectorizer.transform(self._preprocess(X_test)), norm="l2")
        return self.model.predict(X)

    def top_features(self, n=20):
        names = np.array(self.vectorizer.get_feature_names_out())
        coef  = self.model.coef_[0]
        print(f"top {n} -> REAL:")
        for i in np.argsort(coef)[-n:][::-1]:
            print(f"  {names[i]:<30s} {coef[i]:+.4f}")
        print(f"top {n} -> FAKE:")
        for i in np.argsort(coef)[:n]:
            print(f"  {names[i]:<30s} {coef[i]:+.4f}")



BERT Training and Testing

In [ ]:
class FakeNewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.encodings = tokenizer(texts, truncation=True, padding="max_length",
                                   max_length=max_length, return_tensors="pt")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {k: self.encodings[k][idx] for k in ["input_ids", "attention_mask", "token_type_ids"]} | {"labels": self.labels[idx]}


class BERTClassifier:

    def __init__(self, model_name="bert-base-uncased"):
        self.model_name = model_name
        self.tokenizer  = BertTokenizer.from_pretrained(model_name)
        self.model      = None

    def _loader(self, texts, labels, batch_size, shuffle=False):
        return DataLoader(FakeNewsDataset(texts, labels, self.tokenizer),
                          batch_size=batch_size, shuffle=shuffle)

    def train(self, X_train, y_train, batch_size=16, lr=2e-5, max_epochs=4, patience=2):
        print("\n[BERT] fine-tuning...")
        t0 = time.time()

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train
        )
        train_loader = self._loader(X_tr,  y_tr,  batch_size, shuffle=True)
        val_loader   = self._loader(X_val, y_val, batch_size)

        self.model  = BertForSequenceClassification.from_pretrained(self.model_name, num_labels=2).to(DEVICE)
        optimizer   = torch.optim.AdamW(self.model.parameters(), lr=lr)
        total_steps = len(train_loader) * max_epochs
        scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)

        best_f1, best_state, no_improve = 0, None, 0

        for epoch in range(1, max_epochs + 1):
            self.model.train()
            for batch in train_loader:
                optimizer.zero_grad()
                out = self.model(**{k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask", "token_type_ids", "labels"]})
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            vf1 = self._eval_f1(val_loader)
            print(f"  epoch {epoch}  val F1={vf1:.4f}")
            if vf1 > best_f1:
                best_f1, best_state, no_improve = vf1, {k: v.cpu().clone() for k, v in self.model.state_dict().items()}, 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print("  early stopping")
                    break

        self.model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
        elapsed = time.time() - t0
        print(f"[BERT] done ({elapsed:.1f}s)")
        return elapsed

    def _eval_f1(self, loader):
        self.model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in loader:
                out = self.model(**{k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask", "token_type_ids"]})
                preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
                labels.extend(batch["labels"].numpy())
        return f1_score(labels, preds, average="macro")

    def predict(self, texts, batch_size=16):
        self.model.eval()
        preds = []
        with torch.no_grad():
            for batch in self._loader(texts, [0]*len(texts), batch_size):
                out = self.model(**{k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask", "token_type_ids"]})
                preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        return np.array(preds)

    def extract_cls_embeddings(self, texts, batch_size=16):
        self.model.eval()
        embs = []
        with torch.no_grad():
            for batch in self._loader(texts, [0]*len(texts), batch_size):
                out = self.model.bert(**{k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask", "token_type_ids"]})
                embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        return np.vstack(embs)

PLotting

In [ ]:
def plot_cm(y_true, y_pred, model_name, dataset_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'])
    plt.title(f'{model_name} - {dataset_name}')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name}_{dataset_name}.png', dpi=150)
    plt.show()


def plot_comparison(results, dataset_name):
    models = list(results.keys())
    x, w = np.arange(len(models)), 0.35
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(x - w/2, [results[m]['f1'] for m in models],       w, label='Macro-F1', color='steelblue')
    ax.bar(x + w/2, [results[m]['accuracy'] for m in models], w, label='Accuracy',  color='coral')
    ax.set_xticks(x); ax.set_xticklabels(models)
    ax.set_ylim(0, 1.0); ax.set_ylabel('Score')
    ax.set_title(f'Model comparison - {dataset_name}'); ax.legend()
    plt.tight_layout()
    plt.savefig(f'comparison_{dataset_name}.png', dpi=150)
    plt.show()


def plot_ci(results, dataset_name):
    models = list(results.keys())
    f1s    = [results[m]['f1'] for m in models]
    errors = [[f1 - results[m]['ci'][0] for m, f1 in zip(models, f1s)],
              [results[m]['ci'][1] - f1 for m, f1 in zip(models, f1s)]]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(models, f1s, color='steelblue', alpha=0.7, label='Macro-F1')
    ax.errorbar(models, f1s, yerr=errors, fmt='none', color='black', capsize=6, linewidth=2)
    ax.set_ylim(0, 1.0); ax.set_ylabel('Macro-F1')
    ax.set_title(f'Macro-F1 with 95% Bootstrap CI - {dataset_name}'); ax.legend()
    plt.tight_layout()
    plt.savefig(f'ci_{dataset_name}.png', dpi=150)
    plt.show()


def plot_time(results, dataset_name):
    models = list(results.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.bar(models, [results[m]['train_time'] for m in models], color='steelblue')
    ax1.set_ylabel('Training time (s)'); ax1.set_title(f'Training time - {dataset_name}')
    ax2.bar(models, [results[m]['infer_ms'] for m in models], color='coral')
    ax2.set_ylabel('Avg inference (ms/sample)'); ax2.set_title(f'Inference time - {dataset_name}')
    plt.tight_layout()
    plt.savefig(f'time_{dataset_name}.png', dpi=150)
    plt.show()


Running

In [ ]:
LIAR_DIR = '/content/drive/MyDrive/datasets/liar'
FNN_CSV  = '/content/drive/MyDrive/datasets/fakenewsnet/fakenewsnet_politifact.csv'

texts_liar, labels_liar = load_liar(LIAR_DIR)
X_train_liar, X_test_liar, y_train_liar, y_test_liar = split_data(texts_liar, labels_liar)

texts_fnn, labels_fnn = load_fakenewsnet(FNN_CSV)
X_train_fnn, X_test_fnn, y_train_fnn, y_test_fnn = split_data(texts_fnn, labels_fnn)

# SVM
svm_liar = SVMClassifier()
svm_train_time_liar = svm_liar.train(X_train_liar, y_train_liar)
svm_preds = svm_liar.predict(X_test_liar)
evaluate(y_test_liar, svm_preds, 'SVM - LIAR')
svm_liar_ci = bootstrap_ci(y_test_liar, svm_preds)

svm_fnn = SVMClassifier()
svm_train_time_fnn = svm_fnn.train(X_train_fnn, y_train_fnn)
svm_preds_fnn = svm_fnn.predict(X_test_fnn)
evaluate(y_test_fnn, svm_preds_fnn, 'SVM - FakeNewsNet')
svm_fnn_ci = bootstrap_ci(y_test_fnn, svm_preds_fnn)

# BERT
bert_liar = BERTClassifier()
bert_train_time_liar = bert_liar.train(X_train_liar, y_train_liar)
bert_preds_liar = bert_liar.predict(X_test_liar)
evaluate(y_test_liar, bert_preds_liar, 'BERT - LIAR')
bert_liar_ci = bootstrap_ci(y_test_liar, bert_preds_liar)

bert_fnn = BERTClassifier()
bert_train_time_fnn = bert_fnn.train(X_train_fnn, y_train_fnn)
bert_preds_fnn = bert_fnn.predict(X_test_fnn)
evaluate(y_test_fnn, bert_preds_fnn, 'BERT - FakeNewsNet')
bert_fnn_ci = bootstrap_ci(y_test_fnn, bert_preds_fnn)




Saving Training Models

Hybrid Model

In [ ]:
class HybridClassifier:
    def __init__(self, bert_model):
        self.bert = bert_model
        self.svm = None
        self.best_C = None

    def train(self, X_train, y_train):
        t0 = time.time()
        print("\n[Hybrid] fine-tuning BERT...")
        self.bert.train(X_train, y_train)
        print("\n[Hybrid] extracting embeddings...")
        embs = np.asarray(self.bert.extract_cls_embeddings(X_train))
        y_train = np.asarray(y_train)

        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        best_f1, best_C = -1, 1.0

        for C in [0.01, 0.1, 1.0, 10.0]:
            scores = []
            for tr, val in kf.split(embs, y_train):
                clf = LinearSVC(C=C, max_iter=5000, random_state=SEED)
                clf.fit(embs[tr], y_train[tr])
                preds = clf.predict(embs[val])
                scores.append(f1_score(y_train[val], preds, average='macro'))

            m = np.mean(scores)
            print(f"  C={C}  F1={m:.4f}")
            if m > best_f1:
                best_f1, best_C = m, C

        self.best_C = best_C
        self.svm = LinearSVC(C=best_C, max_iter=5000, random_state=SEED)
        self.svm.fit(embs, y_train)

        elapsed = time.time() - t0
        print(f"[Hybrid] done ({elapsed:.1f}s)  best C={best_C}")
        return elapsed

    def predict(self, X_test):
        if self.svm is None:
            raise ValueError("Model has not been trained yet.")

        embs = np.asarray(self.bert.extract_cls_embeddings(X_test))
        return self.svm.predict(embs)

In [ ]:
# LIAR
hybrid_liar = HybridClassifier(BERTClassifier())
hybrid_train_time_liar = hybrid_liar.train(X_train_liar, y_train_liar)
hybrid_preds_liar = hybrid_liar.predict(X_test_liar)
evaluate(y_test_liar, hybrid_preds_liar, 'Hybrid - LIAR')
hybrid_liar_ci = bootstrap_ci(y_test_liar, hybrid_preds_liar)
hybrid_infer_liar = measure_infer_time(hybrid_liar, X_test_liar)

# FakeNewsNet
hybrid_fnn = HybridClassifier(BERTClassifier())
hybrid_train_time_fnn = hybrid_fnn.train(X_train_fnn, y_train_fnn)
hybrid_preds_fnn = hybrid_fnn.predict(X_test_fnn)
evaluate(y_test_fnn, hybrid_preds_fnn, 'Hybrid - FakeNewsNet')
hybrid_fnn_ci = bootstrap_ci(y_test_fnn, hybrid_preds_fnn)
hybrid_infer_fnn = measure_infer_time(hybrid_fnn, X_test_fnn)

In [ ]:
# inference time
svm_infer_liar  = measure_infer_time(svm_liar,  X_test_liar)
svm_infer_fnn   = measure_infer_time(svm_fnn,   X_test_fnn)
bert_infer_liar = measure_infer_time(bert_liar, X_test_liar)
bert_infer_fnn  = measure_infer_time(bert_fnn,  X_test_fnn)

# build results dicts
results_liar = {
    'SVM':  {'f1': f1_score(y_test_liar, svm_preds,      average='macro'), 'accuracy': accuracy_score(y_test_liar, svm_preds),      'ci': svm_liar_ci,  'train_time': svm_train_time_liar,  'infer_ms': svm_infer_liar},
    'BERT': {'f1': f1_score(y_test_liar, bert_preds_liar, average='macro'), 'accuracy': accuracy_score(y_test_liar, bert_preds_liar), 'ci': bert_liar_ci, 'train_time': bert_train_time_liar, 'infer_ms': bert_infer_liar},
}
results_fnn = {
    'SVM':  {'f1': f1_score(y_test_fnn, svm_preds_fnn,  average='macro'), 'accuracy': accuracy_score(y_test_fnn, svm_preds_fnn),  'ci': svm_fnn_ci,  'train_time': svm_train_time_fnn,  'infer_ms': svm_infer_fnn},
    'BERT': {'f1': f1_score(y_test_fnn, bert_preds_fnn, average='macro'), 'accuracy': accuracy_score(y_test_fnn, bert_preds_fnn), 'ci': bert_fnn_ci, 'train_time': bert_train_time_fnn, 'infer_ms': bert_infer_fnn},
}
results_liar['Hybrid'] = {
    'f1':         f1_score(y_test_liar, hybrid_preds_liar, average='macro'),
    'accuracy':   accuracy_score(y_test_liar, hybrid_preds_liar),
    'ci':         hybrid_liar_ci,
    'train_time': hybrid_train_time_liar,
    'infer_ms':   hybrid_infer_liar
}
results_fnn['Hybrid'] = {
    'f1':         f1_score(y_test_fnn, hybrid_preds_fnn, average='macro'),
    'accuracy':   accuracy_score(y_test_fnn, hybrid_preds_fnn),
    'ci':         hybrid_fnn_ci,
    'train_time': hybrid_train_time_fnn,
    'infer_ms':   hybrid_infer_fnn
}
# plots
for name, res, y_test, sp, bp, hp in [
    ('LIAR',        results_liar, y_test_liar, svm_preds,     bert_preds_liar,hybrid_preds_liar),
    ('FakeNewsNet', results_fnn,  y_test_fnn,  svm_preds_fnn, bert_preds_fnn,hybrid_preds_fnn)
]:
    plot_cm(y_test, sp, 'SVM',    name)
    plot_cm(y_test, bp, 'BERT',   name)
    plot_cm(y_test, hp, 'Hybrid', name)
    plot_comparison(res, name)
    plot_ci(res, name)
    plot_time(res, name)


In [ ]:
import shutil
import os


plots = [f for f in os.listdir('/content') if f.endswith('.png')]

os.makedirs('/content/drive/MyDrive/plots1', exist_ok=True)

for p in plots:
    shutil.copy(f'/content/{p}', f'/content/drive/MyDrive/plots1/{p}')
    print(f'saved: {p}')

In [ ]:
# ── SVM top features ──
print("=== SVM top features - LIAR ===")
svm_liar.top_features(n=20)
print("\n=== SVM top features - FakeNewsNet ===")
svm_fnn.top_features(n=20)

# ── LIME predict functions ──
def bert_liar_predict_proba(texts):
    dataset = FakeNewsDataset(texts, [0]*len(texts), bert_liar.tokenizer)
    loader  = DataLoader(dataset, batch_size=16)
    bert_liar.model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            out = bert_liar.model(**{k: batch[k].to(DEVICE)
                                    for k in ["input_ids", "attention_mask", "token_type_ids"]})
            all_logits.append(out.logits.cpu().numpy())
    logits = np.vstack(all_logits)
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def bert_fnn_predict_proba(texts):
    dataset = FakeNewsDataset(texts, [0]*len(texts), bert_fnn.tokenizer)
    loader  = DataLoader(dataset, batch_size=16)
    bert_fnn.model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            out = bert_fnn.model(**{k: batch[k].to(DEVICE)
                                    for k in ["input_ids", "attention_mask", "token_type_ids"]})
            all_logits.append(out.logits.cpu().numpy())
    logits = np.vstack(all_logits)
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def hybrid_liar_predict_proba(texts):
    embs   = hybrid_liar.bert.extract_cls_embeddings(texts)
    scores = hybrid_liar.svm.decision_function(embs)
    probs  = np.column_stack([-scores, scores])
    probs  = 1 / (1 + np.exp(-probs))
    return probs / probs.sum(axis=1, keepdims=True)

def hybrid_fnn_predict_proba(texts):
    embs   = hybrid_fnn.bert.extract_cls_embeddings(texts)
    scores = hybrid_fnn.svm.decision_function(embs)
    probs  = np.column_stack([-scores, scores])
    probs  = 1 / (1 + np.exp(-probs))
    return probs / probs.sum(axis=1, keepdims=True)

# ── LIME analysis ──
from lime.lime_text import LimeTextExplainer
from collections import defaultdict

def run_lime(predict_fn, X_test, model_name, n_samples=200, n_features=10, n_perturb=500):
    explainer = LimeTextExplainer(class_names=['FAKE', 'REAL'])
    indices   = random.sample(range(len(X_test)), min(n_samples, len(X_test)))
    sampled   = [X_test[i] for i in indices]
    weights   = defaultdict(list)
    for i, text in enumerate(sampled):
        exp = explainer.explain_instance(text, predict_fn,
                                         num_features=n_features,
                                         num_samples=n_perturb)
        for word, w in exp.as_list():
            weights[word].append(w)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(sampled)} done...")
    avg    = {w: np.mean(v) for w, v in weights.items()}
    ranked = sorted(avg.items(), key=lambda x: abs(x[1]), reverse=True)
    print(f"\n[LIME - {model_name}] top 20 words:")
    for word, w in ranked[:20]:
        print(f"  {word:<25s} {w:+.4f}  {'-> REAL' if w > 0 else '-> FAKE'}")
    return ranked

print("\n=== LIME - BERT LIAR ===")
lime_bert_liar   = run_lime(bert_liar_predict_proba,   X_test_liar, 'BERT-LIAR')

print("\n=== LIME - BERT FakeNewsNet ===")
lime_bert_fnn    = run_lime(bert_fnn_predict_proba,    X_test_fnn,  'BERT-FNN', n_samples=163)

print("\n=== LIME - Hybrid LIAR ===")
lime_hybrid_liar = run_lime(hybrid_liar_predict_proba, X_test_liar, 'Hybrid-LIAR')

print("\n=== LIME - Hybrid FakeNewsNet ===")
lime_hybrid_fnn  = run_lime(hybrid_fnn_predict_proba,  X_test_fnn,  'Hybrid-FNN', n_samples=163)

In [ ]:
import torch
import transformers
print(torch.__version__)
print(transformers.__version__)
import platform
print(platform.python_version())